In [1]:
import numpy as np
import gymnasium as gym
from collections import defaultdict

In [2]:
class MonteCarloAgent:
    def __init__(self, env, gamma=1.0, epsilon=0.1):
        self.env = env
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = defaultdict(lambda: np.zeros(env.action_space.n))  # Q[state][action]
        self.returns = defaultdict(lambda: defaultdict(list))  # returns[state][action] = list of returns

    def choose_action(self, state):
        if np.random.rand() < self.epsilon:
            return self.env.action_space.sample()
        else:
            return np.argmax(self.Q[state])

    def generate_episode(self):
        episode = []
        state = self.env.reset()[0]
        done = False

        while not done:
            action = self.choose_action(state)
            next_state, reward, done, _ , info = self.env.step(action)
            episode.append((state, action, reward))
            state = next_state

        return episode

    def update(self, episode):
        G = 0
        visited = set()

        # Reverse for return-from-step-t
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = self.gamma * G + reward

            # First-visit MC
            if (state, action) not in visited:
                self.returns[state][action].append(G)
                self.Q[state][action] = np.mean(self.returns[state][action])
                visited.add((state, action))

    def train(self, episodes=10000):
        for i in range(episodes):
            episode = self.generate_episode()
            self.update(episode)
            if (i + 1) % 1000 == 0:
                print(f"Episode {i + 1} completed")

    def get_policy(self):
        policy = np.zeros(self.env.observation_space.n, dtype=int)
        for state in range(self.env.observation_space.n):
            if state in self.Q:
                policy[state] = np.argmax(self.Q[state])
        return policy


In [3]:
def print_policy(policy, shape=(4, 12)):
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    grid = np.array([arrows.get(a, ' ') for a in policy]).reshape(shape)
    grid[3, 0] = 'S'  # Start
    grid[3, 11] = 'G'  # Goal
    for i in range(1, 11):
        grid[3, i] = 'C'  # Cliff
    for row in grid:
        print(" ".join(row))

In [4]:
env = gym.make("CliffWalking-v0")
agent = MonteCarloAgent(env, gamma=1.0, epsilon=0.1)
agent.train(episodes=10000)
policy = agent.get_policy()
print_policy(policy)

Episode 1000 completed
Episode 2000 completed
Episode 3000 completed
Episode 4000 completed
Episode 5000 completed
Episode 6000 completed
Episode 7000 completed
Episode 8000 completed
Episode 9000 completed
Episode 10000 completed
↑ → → → → → → → → ↓ ↓ ↓
↑ ← ← ← ↑ → ↑ → → ↑ ↑ ↓
↑ ← ↓ ← ← ↓ ↑ → ↑ ↑ → ↓
S C C C C C C C C C C G
